# Transfer Learning con mensajes de voz en español
## Clasificación de sentimiento: positivo vs. negativo

**Objetivo didáctico.** Construiremos un clasificador que recibe una grabación de voz en español, transcribe lo que se dijo y clasifica el texto como **positivo** o **negativo**. Compararemos dos estrategias:

1. **Entrenar solo la cabeza:** el modelo lingüístico preentrenado permanece congelado; solo se aprende el clasificador final.
2. **Fine-tuning:** también permitimos que se actualicen los pesos del modelo lingüístico preentrenado.

> **Importante:** este cuaderno estudia el sentimiento expresado por las palabras transcritas, no la emoción acústica de la voz (tono, volumen o prosodia). Por ejemplo, una frase positiva dicha con voz triste podría clasificarse según sus palabras. Para estudiar emociones acústicas necesitaríamos otro modelo y otras etiquetas.

### ¿Qué datos necesitamos?
Necesitamos varias grabaciones cortas, cada una con una etiqueta correcta: `positivo` o `negativo`. Para que la comparación tenga sentido, procure reunir al menos 20–40 audios, idealmente más, equilibrados entre ambas clases. Con muy pocos ejemplos, los resultados son solo una demostración y no permiten concluir qué método es mejor.

Cada audio se transcribirá con Whisper y luego se usará un encoder de español preentrenado (`dccuchile/bert-base-spanish-wwm-cased`). El entrenamiento y la evaluación se harán con las mismas particiones para ambos métodos.

## Paso 1. Preparar el entorno

**¿Qué hace esta celda?** Instala las bibliotecas para transcribir audio, procesar texto y entrenar el modelo. En Colab, elija **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** si está disponible. Si la ejecución da un error de conexión, primero vuelva a conectar el entorno y ejecute esta celda de nuevo.

In [ ]:
!pip -q install transformers datasets accelerate librosa soundfile openai-whisper scikit-learn pandas matplotlib

## Paso 2. Importar bibliotecas y fijar la semilla

**¿Qué hace esta celda?** Importa las herramientas que usaremos y fija una semilla para que la partición de los datos y la inicialización aleatoria sean más reproducibles. La ejecución en GPU acelera el entrenamiento, pero no es obligatoria.

In [ ]:
import os, random, glob, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
)
from datasets import Dataset
from google.colab import files
from IPython.display import Audio, display

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMILLA)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", dispositivo)
print("GPU disponible:", torch.cuda.is_available())

## Paso 3. Cargar las grabaciones

**¿Qué hace esta celda?** Permite seleccionar varios archivos de audio desde el computador. Se recomiendan grabaciones cortas en español (`.wav`, `.mp3`, `.m4a`, etc.). El nombre de cada archivo se usará como identificador; no tiene que contener la etiqueta.

Cargue audios con mensajes positivos y negativos. En el siguiente paso asignaremos manualmente la etiqueta correcta a cada uno.

In [ ]:
subidos = files.upload()
archivos_audio = [nombre for nombre in subidos.keys()
                  if nombre.lower().endswith((".wav", ".mp3", ".m4a", ".ogg", ".flac", ".webm"))]
print(f"Archivos de audio detectados: {len(archivos_audio)}")
for nombre in archivos_audio:
    print(" •", nombre)
if len(archivos_audio) < 4:
    print("Aviso: con menos de 4 audios solo se puede hacer una demostración técnica; se recomienda reunir muchos más.")

## Paso 4. Escuchar los audios y asignar etiquetas

**¿Qué hace esta celda?** Muestra cada grabación y permite escribir su etiqueta real. Las etiquetas deben describir el **sentimiento del contenido verbal**:

- `positivo`: expresa satisfacción, alegría, aprobación o una experiencia favorable.
- `negativo`: expresa insatisfacción, tristeza, rechazo o una experiencia desfavorable.

Si una frase es neutral o ambigua, es mejor no incluirla en esta práctica binaria. La etiqueta es la respuesta correcta que el modelo intentará aprender.

In [ ]:
registros = []
for i, nombre in enumerate(archivos_audio, start=1):
    print(f"\nAudio {i}/{len(archivos_audio)}: {nombre}")
    display(Audio(filename=nombre))
    etiqueta = input("Escriba positivo o negativo: ").strip().lower()
    while etiqueta not in ("positivo", "negativo"):
        etiqueta = input("Etiqueta no válida. Escriba exactamente positivo o negativo: ").strip().lower()
    registros.append({"archivo": nombre, "etiqueta": etiqueta})

df_audio = pd.DataFrame(registros)
display(df_audio)
print("\nConteo por clase:")
print(df_audio["etiqueta"].value_counts())

## Paso 5. Transcribir la voz a texto con Whisper

**¿Qué hace esta celda?** Whisper es un modelo preentrenado para reconocimiento automático del habla. Lo utilizamos para convertir cada grabación en una frase escrita. En esta práctica Whisper **no se entrena**: solamente produce la transcripción que después clasificaremos.

Revise las transcripciones. Si una palabra quedó mal reconocida, corríjala manualmente en la tabla; de lo contrario, el clasificador aprenderá de ese error.

In [ ]:
import whisper

modelo_whisper = whisper.load_model("base")  # modelo multilingüe; primera descarga puede tardar
transcripciones = []
for nombre in df_audio["archivo"]:
    resultado = modelo_whisper.transcribe(nombre, language="es", fp16=torch.cuda.is_available())
    transcripciones.append(resultado["text"].strip())

df = df_audio.copy()
df["texto"] = transcripciones
display(df[["archivo", "texto", "etiqueta"]])

### Revisar o corregir las transcripciones

**¿Qué hace esta celda?** Permite editar las frases transcritas antes del entrenamiento. La transcripción corregida se convierte en la entrada del clasificador.

In [ ]:
# Edite los textos directamente en esta tabla y vuelva a ejecutar la celda.
# Mantenga una fila por audio y no cambie las etiquetas si representan la verdad de referencia.
display(df[["archivo", "texto", "etiqueta"]])
# Ejemplo de corrección manual:
# df.loc[df["archivo"] == "audio1.wav", "texto"] = "La atención fue excelente y estoy muy satisfecho."


## Paso 6. Comprobar si hay suficientes ejemplos

**¿Qué hace esta celda?** Comprueba el número de ejemplos y el equilibrio entre clases. Si una clase tiene muchos más audios que la otra, la exactitud puede ser engañosa. Para una comparación seria, intente tener cantidades parecidas de mensajes positivos y negativos.

In [ ]:
print("Número total de audios:", len(df))
print(df["etiqueta"].value_counts())
if df["etiqueta"].nunique() < 2:
    raise ValueError("Se necesitan ejemplos de las dos clases: positivo y negativo.")
if df["etiqueta"].value_counts().min() < 2:
    raise ValueError("Se necesitan al menos 2 ejemplos por clase para crear una partición estratificada.")
if len(df) < 10:
    print("ADVERTENCIA: hay muy pocos ejemplos. Las métricas serán inestables y solo ilustrativas.")

## Paso 7. Dividir los datos en entrenamiento y prueba

**¿Qué hace esta celda?** Reserva una parte de los audios para la evaluación final. El modelo no verá los ejemplos de prueba durante el entrenamiento. Usaremos exactamente la misma partición para los dos métodos, para que la comparación sea justa.

Con conjuntos muy pequeños, la partición puede ser inestable; idealmente reúna más grabaciones antes de sacar conclusiones.

In [ ]:
etiqueta_a_id = {"negativo": 0, "positivo": 1}
df["label"] = df["etiqueta"].map(etiqueta_a_id).astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SEMILLA, stratify=df["label"]
)
print(f"Entrenamiento: {len(train_df)} audios")
print(f"Prueba: {len(test_df)} audios")
print("\nClases en entrenamiento:")
print(train_df["etiqueta"].value_counts())
print("\nClases en prueba:")
print(test_df["etiqueta"].value_counts())

## Paso 8. Cargar el modelo de lenguaje preentrenado

**¿Qué hace esta celda?** Cargamos BETO, un modelo BERT preentrenado en español. Durante su entrenamiento original aprendió patrones del idioma y representaciones de palabras y frases. Aquí reutilizaremos ese conocimiento para nuestra tarea de sentimiento.

El modelo base no sabe automáticamente qué significa `positivo` o `negativo` en nuestro conjunto específico: por eso entrenaremos una cabeza de clasificación con nuestras etiquetas.

In [ ]:
NOMBRE_MODELO = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO)
print("Tokenizador cargado:", NOMBRE_MODELO)

## Paso 9. Convertir las frases en entradas numéricas

**¿Qué hace esta celda?** El tokenizador divide cada frase en tokens y los convierte en identificadores numéricos. También genera máscaras para indicar qué posiciones corresponden a texto real. Estos números son la entrada del modelo BERT.

No estamos convirtiendo directamente la onda de audio en BETO: primero Whisper transforma voz en texto y luego BETO procesa ese texto.

In [ ]:
def tokenizar_lote(lote):
    return tokenizer(lote["texto"], truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[["texto", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["texto", "label"]].reset_index(drop=True))
train_ds = train_ds.map(tokenizar_lote, batched=True)
test_ds  = test_ds.map(tokenizar_lote, batched=True)
columnas = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=columnas)
test_ds.set_format(type="torch", columns=columnas)
colador = DataCollatorWithPadding(tokenizer=tokenizer)
print("Ejemplo tokenizado:")
print(tokenizer(train_df.iloc[0]["texto"], truncation=True, max_length=128))

## Paso 10. Definir las métricas de evaluación

**¿Qué hace esta celda?** Mediremos exactitud (*accuracy*) y F1 macro. La exactitud indica la proporción total de predicciones correctas. F1 macro calcula el F1 de cada clase y luego promedia ambas, dando el mismo peso a positivo y negativo. Esto ayuda cuando las clases no están perfectamente equilibradas.

In [ ]:
def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicciones),
        "f1_macro": f1_score(labels, predicciones, average="macro", zero_division=0)
    }

argumentos_base = dict(
    output_dir="./resultados_audio_sentimiento",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    report_to="none",
    seed=SEMILLA,
    fp16=torch.cuda.is_available()
)
print("Métricas preparadas: accuracy y F1 macro.")

## Paso 11. Método A: entrenar solo la cabeza de clasificación

**¿Qué hace esta celda?** Cargamos BETO con una cabeza de clasificación de dos clases. Congelamos todos los parámetros del encoder; por tanto, sus representaciones lingüísticas no cambian. Solo se entrenan los pesos de la nueva cabeza final.

Esta estrategia suele ser más rápida y necesita menos memoria. Es una buena primera prueba cuando tenemos pocos datos, aunque el resultado depende de si las características preentrenadas son adecuadas para nuestra tarea.

In [ ]:
modelo_cabeza = AutoModelForSequenceClassification.from_pretrained(
    NOMBRE_MODELO, num_labels=2,
    id2label={0: "negativo", 1: "positivo"},
    label2id={"negativo": 0, "positivo": 1}
)

for parametro in modelo_cabeza.base_model.parameters():
    parametro.requires_grad = False

entrenables = sum(p.numel() for p in modelo_cabeza.parameters() if p.requires_grad)
totales = sum(p.numel() for p in modelo_cabeza.parameters())
print(f"Parámetros entrenables: {entrenables:,} de {totales:,} ({100*entrenables/totales:.2f}%)")

args_cabeza = TrainingArguments(**argumentos_base)
trainer_cabeza = Trainer(
    model=modelo_cabeza,
    args=args_cabeza,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=colador,
    compute_metrics=calcular_metricas
)
trainer_cabeza.train()
metricas_cabeza = trainer_cabeza.evaluate()
print("Resultados — solo cabeza:", metricas_cabeza)

## Paso 12. Método B: fine-tuning de toda la red

**¿Qué hace esta celda?** Volvemos a cargar el modelo desde los pesos preentrenados originales, pero esta vez dejamos entrenables tanto el encoder como la cabeza. Así, las representaciones internas pueden adaptarse al sentimiento de nuestros mensajes.

Usamos una tasa de aprendizaje menor que en la cabeza sola, porque queremos ajustar los conocimientos previos sin destruirlos rápidamente. El fine-tuning puede mejorar el resultado, pero también puede sobreajustarse cuando hay pocos ejemplos.

In [ ]:
modelo_finetuning = AutoModelForSequenceClassification.from_pretrained(
    NOMBRE_MODELO, num_labels=2,
    id2label={0: "negativo", 1: "positivo"},
    label2id={"negativo": 0, "positivo": 1}
)
for parametro in modelo_finetuning.parameters():
    parametro.requires_grad = True

entrenables = sum(p.numel() for p in modelo_finetuning.parameters() if p.requires_grad)
totales = sum(p.numel() for p in modelo_finetuning.parameters())
print(f"Parámetros entrenables: {entrenables:,} de {totales:,} ({100*entrenables/totales:.2f}%)")

argumentos_ft = argumentos_base.copy()
argumentos_ft["learning_rate"] = 2e-5
argumentos_ft["output_dir"] = "./resultados_audio_finetuning"
args_ft = TrainingArguments(**argumentos_ft)
trainer_ft = Trainer(
    model=modelo_finetuning,
    args=args_ft,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=colador,
    compute_metrics=calcular_metricas
)
trainer_ft.train()
metricas_ft = trainer_ft.evaluate()
print("Resultados — fine-tuning:", metricas_ft)

## Paso 13. Comparar ambos métodos

**¿Qué hace esta celda?** Resume las métricas obtenidas en el mismo conjunto de prueba. No hay garantía de que fine-tuning gane: con pocos datos puede rendir peor. Para una evaluación más fiable se necesita un conjunto de datos más amplio y separado de los datos usados para tomar decisiones.

In [ ]:
comparacion = pd.DataFrame([
    {"Método": "Solo cabeza", "Accuracy": metricas_cabeza.get("eval_accuracy"), "F1 macro": metricas_cabeza.get("eval_f1_macro")},
    {"Método": "Fine-tuning", "Accuracy": metricas_ft.get("eval_accuracy"), "F1 macro": metricas_ft.get("eval_f1_macro")}
])
display(comparacion)
ax = comparacion.set_index("Método")[["Accuracy", "F1 macro"]].plot(kind="bar", ylim=(0, 1), figsize=(8, 4))
ax.set_ylabel("Puntuación")
ax.set_title("Comparación en el conjunto de prueba")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.show()

## Paso 14. Examinar errores y matriz de confusión

**¿Qué hace esta celda?** Muestra cómo se comporta cada método por clase. La matriz de confusión ayuda a detectar si el modelo confunde mensajes positivos con negativos. Un resultado agregado alto puede ocultar un desempeño pobre en una de las clases.

In [ ]:
def evaluar_detallado(trainer, nombre):
    salida = trainer.predict(test_ds)
    pred = np.argmax(salida.predictions, axis=1)
    y = salida.label_ids
    print("\n", nombre)
    print(classification_report(y, pred, labels=[0,1], target_names=["negativo","positivo"], zero_division=0))
    cm = confusion_matrix(y, pred, labels=[0,1])
    ConfusionMatrixDisplay(cm, display_labels=["negativo","positivo"]).plot(values_format="d")
    plt.title(f"Matriz de confusión — {nombre}")
    plt.show()
    return pred

pred_cabeza = evaluar_detallado(trainer_cabeza, "Solo cabeza")
pred_ft = evaluar_detallado(trainer_ft, "Fine-tuning")

## Paso 15. Probar una nueva grabación

**¿Qué hace esta celda?** Permite cargar un audio que no formó parte del entrenamiento, transcribirlo con Whisper y clasificar su texto con ambos modelos. La predicción es una estimación del sentimiento expresado por las palabras, no una certeza ni un análisis del tono emocional de la voz.

In [ ]:
nuevos = files.upload()
audio_nuevo = next((n for n in nuevos if n.lower().endswith((".wav", ".mp3", ".m4a", ".ogg", ".flac", ".webm"))), None)
if audio_nuevo is None:
    print("No se encontró un archivo de audio compatible.")
else:
    transcripcion_nueva = modelo_whisper.transcribe(
        audio_nuevo, language="es", fp16=torch.cuda.is_available()
    )["text"].strip()
    print("Transcripción:", transcripcion_nueva)

    def predecir_sentimiento(modelo, texto):
        entradas = tokenizer(texto, return_tensors="pt", truncation=True, max_length=128)
        modelo.eval()
        modelo.to(dispositivo)
        entradas = {k: v.to(dispositivo) for k, v in entradas.items()}
        with torch.no_grad():
            logits = modelo(**entradas).logits
            probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        idx = int(np.argmax(probs))
        return {"clase": ["negativo", "positivo"][idx],
                "P(negativo)": float(probs[0]),
                "P(positivo)": float(probs[1])}

    print("Solo cabeza:", predecir_sentimiento(trainer_cabeza.model, transcripcion_nueva))
    print("Fine-tuning:", predecir_sentimiento(trainer_ft.model, transcripcion_nueva))

## Paso 16. Conclusiones

- **Solo la cabeza:** BETO conserva sus representaciones preentrenadas y aprende únicamente el clasificador final. Normalmente requiere menos recursos y es más rápido.
- **Fine-tuning:** se actualizan también los parámetros de BETO para adaptar las representaciones al conjunto de mensajes. Puede mejorar el resultado, pero requiere más recursos y puede sobreajustarse.
- **La entrada al clasificador es texto transcrito.** Whisper se utiliza como transcriptor preentrenado y no se ajusta en este experimento.
- **No interprete las métricas con pocos audios como una conclusión general.** Los modelos de lenguaje grandes pueden memorizar conjuntos diminutos. Intente reunir más audios variados, balancear las clases y mantener un conjunto de prueba independiente.
- **Posibles mejoras:** validación cruzada, más ejemplos por clase, revisión humana de transcripciones, ajuste de hiperparámetros y evaluación en grabaciones de personas distintas.

### Preguntas para el estudiante
1. ¿Qué parámetros se entrenaron en cada método?
2. ¿Por qué ambos métodos deben usar exactamente la misma partición de prueba?
3. ¿Cuál método obtuvo mayor F1 macro en esta ejecución?
4. ¿Qué podría pasar si casi todos los audios fueran positivos?
5. ¿Qué diferencia habría entre clasificar las palabras transcritas y clasificar la emoción a partir de la señal acústica?